# Project 3: E-Commerce Sales Performance Dashboard
## Data Preprocessing Notebook
**Dataset:** Sample Superstore Dataset (~10,000 rows | 21 columns)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


---
## 1. Load Raw Data

In [4]:
df = pd.read_csv(r'C:\Users\Mohamed\Desktop\dv1\e-commerce\Superstore.csv', encoding='latin1')
print(f'Shape: {df.shape}')
df.head()

Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
# Basic info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

---
## 2. Inspect Data Quality

In [6]:
# Missing values
print('=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

=== Missing Values ===
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

=== Duplicate Rows ===
Duplicates: 0


In [7]:
# Descriptive statistics for numeric columns
df[['Sales', 'Quantity', 'Discount', 'Profit']].describe()

,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,3.789574,0.156203,28.656896
std,623.245101,2.225110,0.206452,234.260108
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.280000,2.000000,0.000000,1.728750
50%,54.490000,3.000000,0.200000,8.666500
75%,209.940000,5.000000,0.200000,29.364000
max,22638.480000,14.000000,0.800000,8399.976000


---
## 3. Data Cleaning

In [8]:
# 3.1 Convert date columns to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')

print('Order Date range:', df['Order Date'].min(), '→', df['Order Date'].max())
print('Ship Date range: ', df['Ship Date'].min(),  '→', df['Ship Date'].max())

Order Date range: 2014-01-03 00:00:00 → 2017-12-30 00:00:00
Ship Date range:  2014-01-07 00:00:00 → 2018-01-05 00:00:00


In [10]:
# 3.3 Ensure numeric columns have correct types
for col in ['Sales', 'Quantity', 'Discount', 'Profit']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows where key numerics are NaN after coercion
before = len(df)
df = df.dropna(subset=['Sales', 'Profit', 'Quantity'])
print(f'Dropped {before - len(df)} rows with invalid numeric values.')

Dropped 0 rows with invalid numeric values.


In [11]:
# 3.4 Validate Discount range (should be 0–1)
print('Discount min:', df['Discount'].min())
print('Discount max:', df['Discount'].max())
# Clip any out-of-range values
df['Discount'] = df['Discount'].clip(0, 1)

Discount min: 0.0
Discount max: 0.8


---
## 4. Feature Engineering

In [12]:
# 4.1 Time-based features
df['Year']    = df['Order Date'].dt.year
df['Month']   = df['Order Date'].dt.to_period('M').astype(str)
df['Quarter'] = df['Order Date'].dt.to_period('Q').astype(str)
df['DayOfWeek'] = df['Order Date'].dt.day_name()

print('Years in dataset:', sorted(df['Year'].unique()))

Years in dataset: [2014, 2015, 2016, 2017]


In [13]:
# 4.2 Derived financial metrics
df['Profit Margin']  = df['Profit'] / df['Sales']          # Ratio
df['Revenue per Unit'] = df['Sales'] / df['Quantity']       # Avg unit price
df['Shipping Days']  = (df['Ship Date'] - df['Order Date']).dt.days

print('Profit Margin stats:')
print(df['Profit Margin'].describe())

Profit Margin stats:
count    9994.000000
mean        0.120314
std         0.466754
min        -2.750000
25%         0.075000
50%         0.270000
75%         0.362500
max         0.500000
Name: Profit Margin, dtype: float64


In [14]:
# 4.3 Discount tiers (categorical binning)
bins   = [-0.001, 0, 0.2, 0.4, 1.0]
labels = ['No Discount', 'Low (1–20%)', 'Medium (21–40%)', 'High (>40%)']
df['Discount Tier'] = pd.cut(df['Discount'], bins=bins, labels=labels)
df['Discount Tier'].value_counts()

Discount Tier
No Discount        4798
Low (1–20%)        3803
High (>40%)         933
Medium (21–40%)     460
Name: count, dtype: int64

---
## 5. Categorical Validation

In [15]:
for col in ['Category', 'Sub-Category', 'Region', 'Segment', 'Ship Mode']:
    print(f'\n{col}: {sorted(df[col].unique())}')


Category: ['Furniture', 'Office Supplies', 'Technology']

Sub-Category: ['Accessories', 'Appliances', 'Art', 'Binders', 'Bookcases', 'Chairs', 'Copiers', 'Envelopes', 'Fasteners', 'Furnishings', 'Labels', 'Machines', 'Paper', 'Phones', 'Storage', 'Supplies', 'Tables']

Region: ['Central', 'East', 'South', 'West']

Segment: ['Consumer', 'Corporate', 'Home Office']

Ship Mode: ['First Class', 'Same Day', 'Second Class', 'Standard Class']


---
## 6. Outlier Inspection

In [16]:
# IQR-based outlier detection for Sales and Profit (for awareness, NOT removed)
for col in ['Sales', 'Profit']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: {len(outliers)} outliers detected (kept for analysis)')

Sales: 1167 outliers detected (kept for analysis)
Profit: 1881 outliers detected (kept for analysis)


---
## 7. Final Dataset Summary

In [17]:
print(f'Final dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
df.head()

Final dataset shape: (9994, 29)

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Year', 'Month', 'Quarter', 'DayOfWeek', 'Profit Margin', 'Revenue per Unit', 'Shipping Days', 'Discount Tier']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Discount,Profit,Year,Month,Quarter,DayOfWeek,Profit Margin,Revenue per Unit,Shipping Days,Discount Tier
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,41.9136,2016,2016-11,2016Q4,Tuesday,0.1600,130.9800,3,No Discount
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,219.5820,2016,2016-11,2016Q4,Tuesday,0.3000,243.9800,3,No Discount
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,0.00,6.8714,2016,2016-06,2016Q2,Sunday,0.4700,7.3100,4,No Discount
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.45,-383.0310,2015,2015-10,2015Q4,Sunday,-0.4000,191.5155,7,High (>40%)
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.20,2.5164,2015,2015-10,2015Q4,Sunday,0.1125,11.1840,7,Low (1–20%)


---
## 8. Export Cleaned Dataset

In [20]:
df.to_csv('../cleaned_data.csv', index=False, encoding='utf-8')
print('Cleaned dataset saved to ../data/cleaned_data.csv')

Cleaned dataset saved to ../data/cleaned_data.csv
